In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv 

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

C:\Users\Mike\AppData\Local\Temp\ipykernel_20060\1850108104.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
import sys
print(sys.executable)

d:\AI Engineering\langchain_agent\.venv\Scripts\python.exe


In [5]:
import langchainhub as hub

In [7]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [8]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [9]:
search_tool = TavilySearchResults(max_results=3)

C:\Users\Mike\AppData\Local\Temp\ipykernel_20060\4144602346.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=3)


In [10]:
result = search_tool.invoke("What is the capital of France?")
result

[{'title': 'What is the capital city of France and why is it famous?',
  'url': 'https://brainly.in/question/62118432',
  'content': 'Aug 13, 2025 — The capital city of France is Paris. Paris is the capital of France, famous for its history, culture, and beauty. Known as the “City of Light,”',
  'score': 0.9257218},
 {'title': 'What is the capital city of France and why is it famous?  | Britannica',
  'url': 'https://www.britannica.com/question/What-is-the-capital-city-of-France-and-why-is-it-famous',
  'content': 'The Editors of Encyclopaedia Britannica\n\nLast updated\n\nOct. 7, 2026\n\nThe capital of France is Paris. Situated in the north-central part of the country, Paris is France’s center of commerce and culture.\n\nFor centuries, Paris has been one of the world’s most prominent cities. Nicknamed the "City of Light" during the Enlightenment, Paris is known for business, commerce, education, culture, and entertainment. The city is appreciated for its gastronomy, haute couture, pai

In [11]:
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0, api_key=GROQ_API_KEY)

In [24]:
from langchain_core.load import loads

In [29]:
from langchainhub import Client
client = Client()
prompt_str = client.pull("hwchase17/react")
prompt = loads(prompt_str) if isinstance(prompt_str, str) else prompt_str

C:\Users\Mike\AppData\Local\Temp\ipykernel_20060\192046012.py:3: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt_str = client.pull("hwchase17/react")
C:\Users\Mike\AppData\Local\Temp\ipykernel_20060\192046012.py:4: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(prompt_str) if isinstance(prompt_str, str) else prompt_str
C:\Users\Mike\AppData\Local\Temp\ipykernel_20060\192046012.py:4: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt = loads(prompt_str) if isinstance(prompt_str, str) else prompt_str


In [30]:
print(prompt)

input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [31]:
tools = [search_tool]

In [32]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [34]:
agent_executer = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [35]:
response = agent_executer.invoke({
    "input": ("What is the capital of Ghana? and what is the name of anthropics current model ?")})



> Entering new AgentExecutor chain...
Question: What is the capital of Ghana? and what is the name of anthropics current model ?
Thought: I need to find two pieces of information: the capital of Ghana and the name of Anthropic's current model. I will search for both.
Action: tavily_search_results_json
Action Input: capital of Ghana[{'title': 'Ghana', 'url': 'https://en.wikipedia.org/wiki/Ghana', 'content': '# Ghana\nThis article is about the country. For the ancient empire, see Ghana Empire. For other uses, see Ghana (disambiguation).\n| Republic of Ghana |\n| --- |\n| Image 6: Flag of Ghana Flag Image 7: Coat of arms of Ghana Coat of arms |\n| Motto:"Freedom and Justice" |\n| Anthem:"God Bless Our Homeland Ghana" Audio 1Duration: 58 seconds.0:58 |\n| Image 8 Image 9 Show globe Show map of Africa Show both |\n| Capital and largest city | Accra Image 10: Show location on an interactive map05°33′18″N 00°11′33″W / 5.55500°N 0.19250°W / 5.55500; -0.19250 |\n| Official languages | English

In [38]:
response["output"]

"The capital of Ghana is Accra. Anthropic's current flagship model is Claude Opus 5.5 (with the most recent release being Claude Haiku 5.5)."